# ManimFlow

Autonomous animation generator powered by [SWE-agent](https://github.com/SWE-agent/SWE-agent) and a hosted [Cloudflare Worker](https://manimator.abbie.workers.dev) — **no API keys required**.

## Usage
1. Runtime → Run All
2. Describe your animation in the text box that appears in the last cell
3. Click **Generate**

In [ ]:
import shutil
if not shutil.which('ffmpeg') or not shutil.which('pkg-config'):
    !apt-get install -y libcairo2-dev libpango1.0-dev ffmpeg 2>&1 | tail -3
else:
    print('system deps already installed')


In [ ]:
import importlib.util, pathlib, sys

if not importlib.util.find_spec('manim'):
    !pip install manim -q

if not pathlib.Path('SWE-agent/pyproject.toml').exists():
    !rm -rf SWE-agent
    !git clone https://github.com/SWE-agent/SWE-agent.git --depth=1 --branch v1.1.0 2>&1 | tail -3

if not importlib.util.find_spec('sweagent'):
    !pip install -e SWE-agent -q

swa_path = str(pathlib.Path('SWE-agent').resolve())
if swa_path not in sys.path:
    sys.path.insert(0, swa_path)

import sweagent
print(f'sweagent ready: {sweagent.__file__}')


In [ ]:
import pathlib, subprocess

output_dir = pathlib.Path('/content/manim_output')
output_dir.mkdir(parents=True, exist_ok=True)

subprocess.run([
    'bash', '-c',
    'cd /content/manim_output && '
    'git init -q && '
    'git config user.email "agent@colab" && '
    'git config user.name "agent" && '
    'git commit --allow-empty -q -m init 2>/dev/null || true'
])

if not pathlib.Path('/manim_output').exists():
    subprocess.run(['ln', '-s', '/content/manim_output', '/manim_output'])

if not pathlib.Path('/content/manim_docs/.git').exists():
    subprocess.run(
        'git clone --depth=1 --filter=blob:none --sparse '
        'https://github.com/ManimCommunity/manim.git /content/manim_docs 2>&1 | tail -2 && '
        'cd /content/manim_docs && git sparse-checkout set docs 2>&1 | tail -1',
        shell=True
    )
    print('Manim docs cloned')
else:
    print('Manim docs already present')

MANIMATOR_CONFIG = """\
agent:
  model:
    name: "openai/manimator"
    api_base: "https://manimator.abbie.workers.dev/v1"
    api_key: "dummy"
    max_input_tokens: 128000
    max_output_tokens: 4096
    per_instance_cost_limit: 0.0
    total_cost_limit: 0.0
  templates:
    system_template: |-
      You are a Manim animation coding agent. You receive a scene plan and must implement it as a working Manim animation.

      DOCS: Manim reference docs are at /content/manim_docs/docs/source/
      - Search: grep -r "ClassName" /content/manim_docs/docs/source/ --include="*.rst" -l
      - Read:   cat /content/manim_docs/docs/source/reference/<file>.rst

      RULES:
      - Use `python3 -m manim` — never `manim`
      - Script goes in /manim_output/scene.py
      - Write files with: python3 -c "import pathlib; pathlib.Path('f.py').write_text(chr(39)*3 + code + chr(39)*3)"
      - Render with: python3 -m manim -pql /manim_output/scene.py <ClassName> 2>&1
      - The scene class must subclass Scene
      - If render fails, read the error and fix — keep trying
      - Stop as soon as you see "File ready at" — output exactly: DONE
      - Do NOT render multiple scenes. ONE script, ONE class, ONE render.
      - Do NOT ask for input. Do NOT explain. Just code and run.

      RESPONSE FORMAT — always exactly this shape, nothing else:
      DISCUSSION
      one line of reasoning
      ```
      one_shell_command
      ```
    instance_template: |-
      Implement this animation plan as a Manim scene named AnimScene in /manim_output/scene.py, then render it.

      PLAN:
      {{problem_statement}}

      Work in: /manim_output
    next_step_template: |-
      OBSERVATION:
      {{observation}}
    next_step_no_output_template: |-
      Command ran with no output.
  tools:
    parse_function:
      type: "thought_action"
    env_variables:
      PAGER: cat
      MANPAGER: cat
      GIT_PAGER: cat
    reset_commands:
      - "set +H"
"""

pathlib.Path('SWE-agent/config/manimator.yaml').write_text(MANIMATOR_CONFIG)
print('Setup complete')


In [ ]:
import subprocess, pathlib, shutil, requests
import IPython.display

WORKER_URL = "https://manimator.abbie.workers.dev/v1/chat/completions"

def call_llm(messages):
    r = requests.post(
        WORKER_URL,
        json={"model": "manimator", "messages": messages, "max_tokens": 1024},
        headers={"Authorization": "Bearer dummy", "Content-Type": "application/json"},
        timeout=60,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"].strip()

def plan_animation(user_prompt):
    return call_llm([
        {
            "role": "system",
            "content": (
                "You are a visual storytelling director. Given a topic or idea, "
                "write a clear scene-by-scene plan for a short educational animation. "
                "Describe what appears on screen, how elements move, and what the viewer sees — "
                "no technical jargon, no code, no tool names. "
                "Be specific: name shapes, colors, labels, and transitions. "
                "Keep it to 3-5 scenes. Format as a numbered list."
            ),
        },
        {"role": "user", "content": user_prompt},
    ])

def run_manimator(user_prompt):
    print(f"Planning: {user_prompt!r}")
    plan = plan_animation(user_prompt)
    print("\n=== Scene Plan ===")
    print(plan)
    print("=" * 40)

    for f in pathlib.Path('/manim_output').glob('*.py'):
        f.unlink()

    task_file = pathlib.Path('/content/task.md')
    task_file.write_text(plan)

    import os
    os.environ['LITELLM_REQUEST_TIMEOUT'] = '90'
    proc = subprocess.Popen(
        [
            "python3", "-m", "sweagent", "run",
            "--config", "SWE-agent/config/manimator.yaml",
            "--env.deployment.type=local",
            "--env.repo.type=preexisting",
            "--env.repo.repo_name=manim_output",
            "--agent.max_requeries=1",
            f"--problem_statement.path={task_file}",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**__import__('os').environ, 'PYTHONPATH': str(pathlib.Path('SWE-agent').resolve())},
        text=True,
        bufsize=1,
    )

    print("\n=== Agent Log ===")
    for line in proc.stdout:
        line = line.rstrip()
        print(line)
        if "File ready at" in line or "Exit status" in line:
            proc.terminate()
            break
    proc.wait()

    videos = sorted(pathlib.Path('/content').rglob('*.mp4'), key=lambda p: p.stat().st_mtime)
    if not videos:
        print("\nNo video rendered.")
        return

    final = videos[-1]
    dest = pathlib.Path('/content/animation.mp4')
    shutil.copy(final, dest)

    for mp4 in videos:
        if mp4 != dest:
            try: mp4.unlink()
            except: pass
    for d in pathlib.Path('/content').rglob('media'):
        if d.is_dir():
            shutil.rmtree(d, ignore_errors=True)

    print(f"\nFinal: {dest}")
    IPython.display.display(
        IPython.display.Video(str(dest), embed=True, html_attributes='controls width="720"')
    )

print("run_manimator() ready")


In [ ]:
import ipywidgets as widgets
from IPython.display import display

prompt_box = widgets.Textarea(
    placeholder="e.g. Explain how a Fourier series builds up a square wave",
    layout=widgets.Layout(width="700px", height="100px"),
)
btn = widgets.Button(description="Generate", button_style="primary")
status = widgets.Output()

def on_generate(b):
    with status:
        status.clear_output()
        if not prompt_box.value.strip():
            print("Please enter a prompt.")
            return
        run_manimator(prompt_box.value.strip())

btn.on_click(on_generate)
display(prompt_box, btn, status)
